Exercise 2.3 (red green blue)

The file src/rgb.txt contains names of colors and their numerical representations in RGB format. The RGB format allows a color to be represented as a mixture of red, green, and blue components. Each component can have an integer value in the range [0,255]. Each line in the file contains four fields: red, green, blue, and colorname. Each field is separated by some amount of whitespace (tab or space in this case). The text file is formatted to make it print nicely, but that makes it harder to process by a computer. Note that some color names can also contain a space character.

Write function red_green_blue that reads the file rgb.txt from the folder src. Remove the irrelevant first line of the file. The function should return a list of strings. Clean-up the file so that the strings in the returned list have four fields separated by a single tab character (\t). Use regular expressions to do this.

The first string in the returned list should be:

'255\t250\t250\tsnow'

In [ ]:

#!/usr/bin/env python3

import re

def red_green_blue(filename="src/rgb.txt"):
    with open(filename) as in_file:
        l = re.findall(r"(\d+)\s+(\d+)\s+(\d+)\s+(.*)\n", in_file.read())
        return [
            "{}\t{}\t{}\t{}".format(r, g, b, name)
            for r, g, b, name
            in l
        ]


def main():
    red_green_blue()

if __name__ == "__main__":
    main()

## Simple Walkthrough — Line by Line

Let's trace exactly what happens, step by step, using a tiny example file to make it concrete.

**Imagine `rgb.txt` looks like:**
```
red   green   blue    name

255   250   250       snow
  0     0     0       black
```

---

### Step 1 — Open the File

```python
with open(filename) as in_file:
```

Standard file opening — you've done this many times. `with` auto-closes it when done.

---

### Step 2 — Read the WHOLE File as One Big String

```python
in_file.read()
```

Unlike looping line-by-line, `.read()` slurps the **entire file** into a single string, newlines and all:

```python
"red   green   blue    name\n\n255   250   250       snow\n  0     0     0       black\n"
```

One long string — no separate "lines" concept anymore, just one blob of text with `\n` characters scattered inside it.

---

### Step 3 — `findall` Scans That Whole Blob for Matches

```python
re.findall(r"(\d+)\s+(\d+)\s+(\d+)\s+(.*)\n", in_file.read())
```

The pattern says: *"find every place with digits, space, digits, space, digits, space, then anything, then a newline."*

`findall` doesn't care about line boundaries — it just **scans left to right through the entire blob**, and every time it finds text matching that shape, it records it.

```
"red   green   blue    name\n\n255   250   250       snow\n  0     0     0       black\n"
                                └──┬──┘  └┬─┘  └┬─┘       └──┬──┘
                                digit  digit digit        name  ← MATCH found here!
```

The header line `"red   green   blue    name"` has **no digits** — so the pattern simply can't start matching there. `findall` glides right past it and finds the **first real match** starting at `255`.

---

### Step 4 — Because There Are 4 Groups, Each Match Becomes a TUPLE

Remember your earlier lesson: *"with multiple groups, `findall` returns tuples of `.groups()`."* That's exactly what happens here:

```python
l = [
    ('255', '250', '250', 'snow'),
    ('0', '0', '0', 'black'),
]
```

One tuple per matched line, with 4 strings inside each.

---

### Step 5 — The List Comprehension Rebuilds Each Tuple Into a Tab-Joined String

```python
return [
    "{}\t{}\t{}\t{}".format(r, g, b, name)
    for r, g, b, name
    in l
]
```

This is a comprehension with **tuple unpacking**, just like the `for name, age in ...` pattern from your dictionary/regex lessons. For each tuple in `l`, it unpacks the four values into `r`, `g`, `b`, `name`, then builds a formatted string:

```
('255', '250', '250', 'snow')   →   "255\t250\t250\tsnow"
('0', '0', '0', 'black')        →   "0\t0\t0\tblack"
```

`.format(r, g, b, name)` fills the `{}` placeholders in order — the same `.format()` method from way back in your formatting lessons.

---

### The Final Result

```python
["255\t250\t250\tsnow", "0\t0\t0\tblack"]
```

Exactly the shape the exercise wants — a list of tab-separated strings, header excluded, without you writing any explicit "skip line 1" code.

---

### The Whole Pipeline, Visualized

```
FILE ON DISK
    ↓  open() + .read()
ONE BIG STRING
    ↓  re.findall(pattern, ...)
LIST OF TUPLES     [('255','250','250','snow'), ('0','0','0','black')]
    ↓  comprehension + .format()
LIST OF STRINGS    ['255\t250\t250\tsnow', '0\t0\t0\tblack']
```

Three transformations, each one a tool you already know individually — just chained together.

---

### Why This Skips the Header "For Free"

The trick worth remembering: the header has **no digits**, and the pattern **starts** with `\d+` (mandatory digit). A pattern can only match text that actually fits its shape — since `"red   green   blue    name"` has zero digits anywhere in it, there's **nothing for `\d+` to grab onto**, so that entire line is invisible to the pattern. No explicit skip-logic needed; the requirement is baked into what counts as a "match" at all.

---

### The One-Sentence Summary

> Read the whole file as one string → let `findall` hunt for every occurrence of the 4-field digit/digit/digit/name shape anywhere in it (automatically skipping the non-numeric header) → each match becomes a 4-tuple → unpack each tuple in a comprehension and rebuild it as a tab-joined string. Three familiar tools (`.read()`, `findall` with groups, comprehension+`.format()`) chained into one compact pipeline. 🎯